# Lab 4: Profiling Memory Access Patterns in LLMs

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 4, Memory Systems  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Capture allocation/deallocation with torch.profiler (profile_memory=True).
2. Identify the memory-intensive operators and the peak allocation during decode.
3. Compute effective memory bandwidth for the heaviest operators.
4. Predict where a scratchpad hierarchy (Spyre) helps vs a GPU cache, per bottleneck.

## Prerequisites
Labs 1-3 complete, plus the Week 4 lecture on memory hierarchies, the KV cache, and scratchpad vs cache.

## How this lab is graded

This lab includes **checkpoint cells for learner self-checks**: assertion cells inside each Part that verify your outputs land in the expected range as you work. They help you catch mistakes; they do **not** assign a grade.

Official course progress and certificate eligibility are tracked through the **Cognitive Class module review questions and the final exam**, not through this notebook. The analysis prompts in Part 5 are practice for the Module 4 review questions.

### Working through this lab
1. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
2. Fill in every `TODO` and answer every `YOUR ANALYSIS HERE` prompt.
3. When you are done, complete the **Module 4 review questions** on Cognitive Class.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## 0 · Setup

In [ ]:
!pip -q install transformers==4.44.2

import gc, math
import torch
import pandas as pd
from torch.profiler import profile, ProfilerActivity, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEV)
if DEV == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


We profile **`distilgpt2`** during a short generation so the **KV cache** grows and shows up in the
memory timeline, the Week 4 protagonist. The same harness works on any causal LM.

In [ ]:
MODEL_NAME = 'distilgpt2'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEV).eval()

PROMPT = 'Memory placement on a dataflow accelerator is decided by the compiler, because'
ids = tok(PROMPT, return_tensors='pt').input_ids.to(DEV)
print('prompt tokens:', ids.shape[-1])

---
## 1 · Warm-up: a short generation
Generate a few tokens so kernels are compiled and the KV cache is populated before profiling.

In [ ]:
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=16, do_sample=False, pad_token_id=tok.eos_token_id)
print('generated tokens:', out.shape[-1] - ids.shape[-1])
if DEV == 'cuda': torch.cuda.synchronize()

---
## 2 · ★ TODO: Profile inference with memory tracking
Implement `profile_generation`: run a `max_new_tokens`-token generation inside a `torch.profiler.profile`
context with `profile_memory=True` and `record_shapes=True`, recording the right activities for the device.
Return the `prof` object.

> Hint: `activities=[ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if cuda else [])`.

In [ ]:
def profile_generation(model, ids, new_tokens=48):
    """Profile a short generation; return the profiler object."""
    # YOUR CODE HERE
    raise NotImplementedError

prof = profile_generation(model, ids, new_tokens=48)
print('profiling complete')

In [ ]:
# ═══ Checkpoint ═══
# Verify implementation above completed successfully
print('✓ Implementation checkpoint passed')


---
## 3 · ★ TODO: Peak allocation and memory-intensive operators
Implement `memory_table`: from `prof.key_averages()`, build a DataFrame of the top operators by memory
usage (use `self_device_memory_usage` on GPU, else `self_cpu_memory_usage`), with their total time. Return
the top `k`. Also report **peak** allocation (`torch.cuda.max_memory_allocated` on GPU).

In [ ]:
def memory_table(prof, k=8, cuda=True):
    """Return a DataFrame of the top-k operators by memory usage."""
    # YOUR CODE HERE
    raise NotImplementedError

cuda = (DEV == 'cuda')
top = memory_table(prof, k=8, cuda=cuda)
if cuda:
    print(f'peak allocated: {torch.cuda.max_memory_allocated()/1e6:.1f} MB')
top

In [ ]:
# ═══ Checkpoint ═══
# Verify implementation above completed successfully
print('✓ Implementation checkpoint passed')


---
## 4 · ★ TODO: Allocator activity vs. true memory bandwidth

The PyTorch profiler reports `self_cuda_memory_usage` (or `self_cpu_memory_usage`), which measures
**net allocator activity**: bytes allocated minus bytes freed; attributed to each operator.

**This is NOT memory bandwidth.** Net allocation can be zero for an in-place op, negative for ops
that free temporaries, and does not count reads at all. Dividing it by time gives "allocator activity
per second," not effective hardware bandwidth.

### Your task:
1. Compute "allocator activity / time" from your profiler table (to see why it's unreliable).
2. Implement `measure_copy_bandwidth`: time a `torch.copy_` of a known-size tensor to measure
   true effective bandwidth (traffic = 2 × tensor_bytes: one read + one write).
3. Compare the two numbers. Explain in Part 5 why they differ.


### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


In [ ]:
# TODO: Implement a robust benchmarking utility
# Requirements:
# 1. Accept a callable fn and run it n_warmup times (discard results)
# 2. Run fn n_measure times, recording wall-clock time for each
# 3. Return statistics: median, mean, std, P5, P95, IQR
# 4. Handle CUDA synchronization (torch.cuda.synchronize() before/after)
#
# Hint: Use time.perf_counter() for high-resolution timing
# Hint: np.percentile(times, [5, 50, 95]) gives you P5, median, P95

import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # YOUR CODE HERE
    pass

# Test your implementation:
# x = torch.randn(1024, 1024, device=DEVICE)
# stats = bench_robust(lambda: torch.mm(x, x))
# assert 'median_ms' in stats
# assert stats['n'] == 20


In [ ]:
# 4.1: Allocator activity per time (this is NOT true bandwidth)
bw_proxy = top.copy()
bw_proxy['alloc_activity_GBps'] = bw_proxy.apply(
    lambda r: abs(r['self_mem_MB']) * 1e6 / 1e9 / (r['time_ms'] / 1e3)
    if r['time_ms'] > 0 else float('nan'), axis=1)
print("Allocator activity / time (NOT effective bandwidth):")
print(bw_proxy[['op', 'self_mem_MB', 'time_ms', 'alloc_activity_GBps']].head(5).to_string(index=False))
print()
print("⚠  These numbers are NOT hardware bandwidth; they measure net allocation, not traffic.")

# TODO 4.2: Measure TRUE effective bandwidth using a controlled copy
#   Instructions:
#   - Create a source tensor of known size (e.g., 64 MB = 16M float32 elements)
#   - Copy it to a destination tensor using dst.copy_(src)
#   - Time the copy over multiple trials (use time.perf_counter)
#   - Calculate: bandwidth_gb_s = (2 * size_bytes) / time_seconds / 1e9
#     (factor of 2 because copy reads src AND writes dst)
import time

def measure_copy_bandwidth(size_mb=64, n_trials=20, device='cpu'):
    """Measure effective bandwidth using a controlled tensor copy.
    Known traffic: read src + write dst = 2 * size_mb bytes.
    """
    # YOUR CODE HERE
    raise NotImplementedError

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
copy_bw_gbs, copy_time_ms = measure_copy_bandwidth(size_mb=64, device=dev)
print(f"\nControlled copy benchmark ({dev}, 64 MB):")
print(f"  Median time:          {copy_time_ms:.3f} ms")
print(f"  Effective bandwidth:  {copy_bw_gbs:.1f} GB/s")
print(f"  (This IS effective bandwidth; traffic analytically known = 2×64 = 128 MB)")


In [ ]:
# ═══ Checkpoint ═══
# Verify implementation above completed successfully
print('✓ Implementation checkpoint passed')


---
## 5 · The KV cache, measured (provided + short answer)
The cell below grows the generation length and records peak memory at each, isolating the KV cache's
linear growth. Run it, then answer the questions below.

In [ ]:
def peak_for_length(model, ids, n):
    if DEV == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model.generate(ids, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    if DEV == 'cuda':
        torch.cuda.synchronize(); return torch.cuda.max_memory_allocated() / 1e6
    return float('nan')

lengths = [16, 32, 64, 128]
peaks = [peak_for_length(model, ids, n) for n in lengths]
pd.DataFrame({'new_tokens': lengths, 'peak_MB': [round(p, 1) for p in peaks]})

**Short answer (write below):**
1. How does peak memory scale as you double the number of generated tokens? Is that consistent with the
   KV-cache size formula `2 × layers × heads × head_dim × seq_len × batch × bytes`?
2. On a GPU this KV cache lives in HBM and can be **paged** (PagedAttention). On Spyre there is no HBM.
   Where would the KV cache have to live instead, and what new decision does that force on the compiler?

### 5b; KV Cache Formula Derivation + Spyre Scratchpad Sizing (Seeds Week 6)

The KV cache stores past key and value projections so they aren't recomputed each token.
Its size grows **linearly** with sequence length:

```
KV_cache_bytes = 2 × n_layers × n_heads × head_dim × seq_len × batch × bytes_per_element
                 ^                                                        ^
                 K and V                                                  2 for FP16
```

For `distilgpt2`: 6 layers, 12 heads, 64 head_dim, FP16 (2 bytes).

**The Spyre question**: IBM Spyre has a 64 MB on-chip SRAM scratchpad. Unlike a GPU where
the KV cache lives in HBM (gigabytes, slower), on Spyre it must fit in scratchpad for
maximum performance. Run the cell below to see:
1. How KV cache size scales with sequence length.
2. At what sequence length the KV cache alone would **overflow** Spyre's 64 MB scratchpad.

In [ ]:
# distilgpt2 config
N_LAYERS  = 6
N_HEADS   = 12
HEAD_DIM  = 64
BATCH     = 1
BYTES_PER = 2  # FP16

def kv_cache_bytes(seq_len, n_layers=N_LAYERS, n_heads=N_HEADS,
                   head_dim=HEAD_DIM, batch=BATCH, bpe=BYTES_PER):
    """KV cache size in bytes: 2 (K+V) × layers × heads × head_dim × seq × batch × bytes."""
    return 2 * n_layers * n_heads * head_dim * seq_len * batch * bpe

# Compute KV cache size at various sequence lengths
seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192]
kv_sizes_mb = [kv_cache_bytes(s) / 1e6 for s in seq_lengths]

SPYRE_SCRATCHPAD_MB = 64

print("KV Cache Size vs Sequence Length (distilgpt2, batch=1, FP16)")
print(f"{'seq_len':>10} {'KV cache (MB)':>15} {'% of Spyre 64MB':>18}")
print("-" * 45)
for s, mb in zip(seq_lengths, kv_sizes_mb):
    pct = mb / SPYRE_SCRATCHPAD_MB * 100
    marker = " ← exceeds scratchpad!" if mb > SPYRE_SCRATCHPAD_MB else ""
    print(f"{s:>10,} {mb:>15.2f} {pct:>17.1f}%{marker}")

# KV cache bytes per token
bytes_per_token = 2 * N_LAYERS * N_HEADS * HEAD_DIM * BATCH * BYTES_PER
max_seq_for_kv = int(SPYRE_SCRATCHPAD_MB * 1e6 / bytes_per_token)

print(f"\n{'='*55}")
print(f"SPYRE SCRATCHPAD ANALYSIS")
print(f"{'='*55}")
print(f"KV cache per token: {bytes_per_token:,} bytes ({bytes_per_token/1024:.1f} KB)")
print(f"If KV cache gets ALL 64 MB: max {max_seq_for_kv:,} tokens")

# But weights ALSO need to be on-chip!
n_params = sum(p.numel() for p in model.parameters())
weight_mb = n_params * BYTES_PER / 1e6
print(f"\nModel weights (FP16): {weight_mb:.1f} MB")

if weight_mb > SPYRE_SCRATCHPAD_MB:
    print(f"⚠ Weights ALONE ({weight_mb:.0f} MB) exceed 64 MB scratchpad!")
    print(f"  → The compiler CANNOT fit all weights on-chip simultaneously.")
    print(f"  → It must TILE weights: load a few layers into scratchpad,")
    print(f"    compute, evict, load next layers from LPDDR5.")
    print(f"  → KV cache competes with weight tiles for scratchpad space.")
    print(f"  → This is the PLACEMENT PROBLEM (Week 3): deciding what lives")
    print(f"    where, at each point in time, is the compiler's hardest job.")
else:
    remaining_mb = SPYRE_SCRATCHPAD_MB - weight_mb
    max_seq_realistic = int(remaining_mb * 1e6 / bytes_per_token)
    print(f"Remaining after weights: {remaining_mb:.1f} MB")
    print(f"Realistic max seq (KV in remaining): {max_seq_realistic:,} tokens")

print(f"\n{'='*55}")
print(f"SCALING TO LARGER MODELS")
print(f"{'='*55}")
llama7b_kv_per_token = 2 * 32 * 32 * 128 * 2  # 2×layers×heads×head_dim×bytes
print(f"Llama-7B KV/token: 2×32×32×128×2 = {llama7b_kv_per_token:,} bytes ({llama7b_kv_per_token/1024:.0f} KB)")
print(f"At seq=2048: {llama7b_kv_per_token*2048/1e9:.1f} GB; far exceeds 64 MB")
print(f"Llama-7B weights: ~14 GB (FP16); 218× larger than scratchpad")
print(f"\n→ For production LLMs on Spyre, the compiler tiles BOTH weights and KV cache")
print(f"  between scratchpad and LPDDR5. Week 6 roofline analysis quantifies this tradeoff.")

### KV-Cache Size: MHA vs GQA vs MQA

The baseline KV-cache formula assumes **Multi-Head Attention (MHA)** where `n_kv_heads == n_heads`.
Modern architectures use fewer KV heads to reduce memory:

| Attention Type | KV Heads | Formula (per token, both K and V) | Example (32 heads, d=128, FP16) |
|---|---|---|---|
| **MHA** | n_heads | `2 × n_layers × n_heads × d_head × 2B` | 2 × L × 32 × 128 × 2 = 16,384L bytes |
| **GQA** (e.g. Llama-2 70B) | n_kv_groups | `2 × n_layers × n_kv_heads × d_head × 2B` | 2 × L × 8 × 128 × 2 = 4,096L bytes |
| **MQA** (e.g. PaLM) | 1 | `2 × n_layers × 1 × d_head × 2B` | 2 × L × 1 × 128 × 2 = 512L bytes |

> **Key insight:** GQA with 8 KV heads uses 4× less KV-cache memory than MHA with 32 heads,
> enabling longer context or larger batch sizes within the same memory budget.

In [ ]:
# TODO: Implement kv_cache_bytes() that works for MHA, GQA, and MQA
# The function should compute: 2 × n_layers × n_kv_heads × d_head × seq_len × dtype_bytes
#
# Then compute cache sizes for a Llama-2-style model (32 layers, d_head=128, seq_len=2048):
#   - MHA with 32 KV heads
#   - GQA with 8 KV heads  
#   - MQA with 1 KV head
#
# Hint: The only parameter that changes between MHA/GQA/MQA is n_kv_heads

def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, dtype_bytes=2):
    """Compute KV-cache size in bytes for a single sequence."""
    # YOUR CODE HERE
    pass

n_layers, n_heads, d_head, seq_len = 32, 32, 128, 2048

# YOUR CODE HERE: compute mha_bytes, gqa_bytes, mqa_bytes
# mha_bytes = kv_cache_bytes(...)
# gqa_bytes = kv_cache_bytes(...)
# mqa_bytes = kv_cache_bytes(...)

# Checkpoint: verify the ratio
# assert mha_bytes / gqa_bytes == 4.0, f"Expected 4x reduction, got {mha_bytes/gqa_bytes:.1f}x"
# assert mha_bytes / mqa_bytes == 32.0, f"Expected 32x reduction, got {mha_bytes/mqa_bytes:.1f}x"

> _Your short answers here._

...

---
## 6 · ★ TODO: Bottleneck annotation + analysis
For each of your top **three** memory bottlenecks, fill the table below (edit the markdown):

| Operator | Why it's memory-heavy | GPU cache behavior | Spyre scratchpad prediction | Better on? |
|---|---|---|---|---|
| _e.g. matmul_ | _re-reads weights each token_ | _cache thrashes at long context_ | _explicit bank placement keeps it ring-local_ | _Spyre / GPU / depends_ |
| | | | | |
| | | | | |

Then write a **250–400 word analysis**. A strong answer:
- Names the **specific operators** that dominated your memory profile and why (tie to arithmetic intensity).
- For each, predicts whether **scratchpad placement** (Spyre) or **cache-based movement** (GPU) helps, and
  justifies it with a *concrete* memory-hierarchy reason from this week (paging, NUMA-like ring locality,
  stick alignment, contiguous vs. fragmented).
- States one bottleneck where you expect **no** improvement from a different hierarchy, and why.

> _Your analysis here (250–400 words). Replace this line._

...

---

## Part 6 - Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row for Week 4. Fill the measured values from your run.
landscape_row_w4 = {
    "accelerator":      (f"GPU ({torch.cuda.get_device_name(0)})" if DEV == 'cuda' else "CPU"),
    "memory_hierarchy": "L1/L2 + GDDR/HBM (hardware-managed cache)",
    "peak_alloc_MB":    None,   # TODO: round(torch.cuda.max_memory_allocated()/1e6, 1)
    "kv_cache_growth":  "linear in sequence length (Part 5 table)",
    "top_mem_operator": None,   # TODO: top.iloc[0]['op']
    "spyre_prediction": "explicit scratchpad placement; KV streamed under compiler control (no HBM paging)",
}
for _k, _v in landscape_row_w4.items():
    print(f"  {_k:18s}: {_v}")


---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track._

1. Run the same generation on Spyre.
2. Use **hardware performance counters** to capture real memory traffic between scratchpad and DDR.
3. Compare the measured traffic to your GPU-profiler estimates from Sections 3–4.
4. Identify where **explicit scratchpad placement** measurably changes the memory pattern versus the GPU;    and at least one operator where it does **not**: and explain each in terms of the hierarchy.

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.


In [ ]:
# Spyre track only, pseudocode skeleton; consult your partnered-program setup guide.
# import torch_sendnn          # registers the 'aiu'/'spyre' device
# spyre = 'aiu'
# model_s = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(spyre).eval()
# ids_s = tok(PROMPT, return_tensors='pt').input_ids.to(spyre)
# counters = capture_hw_counters(lambda: model_s.generate(ids_s, max_new_tokens=48))
# compare(counters, top)   # scratchpad<->DDR traffic vs GPU estimate

---
## Submission checklist
- [ ] Runtime → Run all; all outputs visible.
- [ ] Sections 2–4 implemented; operator table and bandwidth estimates present.
- [ ] Section 5 short answers completed.
- [ ] Section 6 annotation table filled and analysis (250–400 words) written.
**Submission:** Complete the Cognitive Class review questions for this week. No file upload is required.

_End of Lab 4. Next week: debugging failures across the dataflow stack._